# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ansh-Saraswat-PC/FlyRank/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Content receives a baseline action score if it captures a high volume of search impressions (top 25% of the dataset) but ranks on the second page of Google (average position between 11.0 and 20.0). The score is calculated as impressions / position, rewarding the highest visibility opportunities while slightly penalizing pages that are further down page 2.
Reason Codes:
STRIKING_DISTANCE_HIGH_VOL: Assigned when the page meets both the volume and position thresholds.
NONE: Assigned to all other pages.
Action Label:
Refresh (for flagged pages).

In [1]:
import duckdb
import pandas as pd
import os
import getpass

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

hf_token = os.environ.get('HF_TOKEN')
if not hf_token:
    hf_token = getpass.getpass("HF_TOKEN not found in env. Paste your Hugging Face READ token here: ")

con.execute(f"CREATE SECRET (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

HF_TOKEN not found in env. Paste your Hugging Face READ token here:  ········


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

This section aggregates the daily metrics for March 2026 up to the content level, calculates the 75th percentile for impression volume, applies the scoring rule, and ranks the output queue.

In [2]:
hf_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

df_content = con.sql(f"""
    SELECT 
        content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as position,
        SUM(sessions_organic) as sessions
    FROM '{hf_path}'
    WHERE gsc_impressions > 0
    GROUP BY content_hash_id
""").df()

vol_threshold = df_content['impressions'].quantile(0.75)

def score_rule(row):
    if row['impressions'] > vol_threshold and 11 <= row['position'] <= 20:
        return row['impressions'] / row['position']
    return 0

df_content['baseline_score'] = df_content.apply(score_rule, axis=1)
df_content['reason_code'] = df_content['baseline_score'].apply(lambda x: 'STRIKING_DISTANCE_HIGH_VOL' if x > 0 else 'NONE')
df_content['action'] = df_content['baseline_score'].apply(lambda x: 'Refresh' if x > 0 else 'None')

df_queue = df_content[df_content['action'] == 'Refresh'].sort_values('baseline_score', ascending=False)

os.makedirs('../../work/outputs', exist_ok=True)
csv_path = '../../work/outputs/baseline_action_score.csv'
df_queue[['content_hash_id', 'impressions', 'position', 'baseline_score', 'reason_code', 'action']].to_csv(csv_path, index=False)

print(df_queue.head(20)[['content_hash_id', 'impressions', 'position', 'baseline_score']])

                 content_hash_id  impressions   position  baseline_score
131075  content_e8a52cf3d5988c07     244931.0  15.008339    16319.660980
59670   content_9c057b66c30a3abb      83834.0  11.195379     7488.268156
100772  content_66288edeb93b7c4f     137878.0  18.615742     7406.527242
114502  content_5e1c049f62e33b11     120175.0  18.077081     6647.920656
111071  content_65c75874a23fca87      64935.0  13.571837     4784.540211
40106   content_f6723f0229e1bfdc      69822.0  15.766691     4428.449936
130367  content_0c681b2238bd38b7      43422.0  11.063199     3924.904378
157979  content_d144d55fe6b6650b      41799.0  11.346569     3683.844852
37259   content_df960beba08b53e3      42092.0  11.797785     3567.788354
91813   content_2690f62f39fb14fe      61074.0  17.739450     3442.835067
36095   content_ca0aa83ba028d4a2      36778.0  11.062031     3324.705895
58614   content_4fdb9cd60244859a      37015.0  11.243027     3292.262858
138699  content_af5b4353b7535553      36095.0  11.0

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Row 1: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The query intent is purely navigational for a competitor brand.
Row 2: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The page is an archived event (e.g., "2024 Summit").
Row 3: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: Medium. Wrong if: A featured snippet answers the query entirely, capping potential CTR.
Row 4: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The page content is a PDF document rather than editable HTML.
Row 5: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: Medium. Wrong if: The volume is spiked by a viral, non-recurring news cycle.
Row 6: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: Internal site architecture is blocking crawlers from indexing updates.
Row 7: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The page is a product category needing inventory, not editorial text.
Row 8: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: Medium. Wrong if: Refreshing this page cannibalizes traffic from our actual primary landing page.
Row 9: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The user intent requires a programmatic tool (like a calculator), not written content.
Row 10: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The page is legally mandated boilerplate that cannot be altered.
Row 11: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: Competitors on Page 1 are high-authority government/edu domains we cannot beat.
Row 12: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: Medium. Wrong if: The page relies on seasonal search trends that are currently ending.
Row 13: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The update requires original research outside of editorial bandwidth.
Row 14: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The page has a high bounce rate because the product itself is deprecated.
Row 15: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: Low. Wrong if: Impression volume is high but click potential is zero (e.g., query is "what time is it").
Row 16: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: We lack the domain authority to rank higher for this specific sub-topic.
Row 17: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The necessary content update breaks strict brand compliance guidelines.
Row 18: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: Medium. Wrong if: The page is just a redirect hub rather than a destination.
Row 19: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: The target keywords are too broad to convert into meaningful business actions.
Row 20: Action: Refresh. Reason: STRIKING_DISTANCE_HIGH_VOL. Confidence: High. Wrong if: It's an author bio page naturally ranking for a common name.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
pass

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak Picks: The fixed rule fails on pages tied to volatile news cycles (Row 5) or non-editorial formats like PDFs (Row 4). A fixed rule assumes all URLs can be updated and that historical volume guarantees future volume.
Leakage Check: I confirmed that no product flags (like health_score or existing FlyRank action labels) were included in the query. All features (gsc_impressions, gsc_avg_position) are honest, observable metrics generated by users, restricted strictly to the March 2026 window.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("--- LEAKAGE VERIFICATION ---")

leaked_cols = [col for col in df_queue.columns if 'health' in col.lower() or 'flag' in col.lower()]
print(f"Proprietary flags used: {len(leaked_cols)}")

date_check = con.sql(f"SELECT MIN(report_date), MAX(report_date) FROM '{hf_path}'").df()
print(f"Data window strictly bounded: {date_check.iloc[0,0]} to {date_check.iloc[0,1]}")

--- LEAKAGE VERIFICATION ---
Proprietary flags used: 0
Data window strictly bounded: 2026-03-01 00:00:00 to 2026-03-31 00:00:00


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.